# Discover CNN

Nguyễn Ngọc Hoàng Nam - B23DCCN585 | Assignment 04

**Bản mở rộng:** notebook này giữ nhóm 18 lượt seed 42 để giải thích thuật toán. Notebook **06** tổng hợp đầy đủ 54 lượt nhiều seed và 18 ablation; notebook **07** thực thi ví dụ số học dùng trong báo cáo 78 trang.

In [1]:
from pathlib import Path
import os, sys, json, subprocess
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT / 'src').is_dir(), 'Hãy mở notebook từ thư mục repository.'
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from IPython.display import display, Markdown, Image
from src.data import DATASETS, load_data, data_root
print('Python:', sys.version.split()[0])
print('Dữ liệu:', data_root())

Python: 3.10.20
Dữ liệu: E:\PTHTTM\ASG_04_data


## 1. CNN là phép hợp thành hàm

Một mạng biến ảnh $x$ thành logits $z = f_L(\cdots f_2(f_1(x)))$. Softmax chuyển logits thành phân phối xác suất. Convolution học các bộ lọc cục bộ; ReLU tạo phi tuyến; pooling giảm kích thước; dense thực hiện phân loại. Nếu bỏ mọi phi tuyến giữa các phép biến đổi tuyến tính, các lớp tuyến tính liên tiếp vẫn có thể gộp thành một phép biến đổi tuyến tính.

Trọng số được chia sẻ giữa các vị trí ảnh, giúp giảm số tham số so với nối đầy đủ trực tiếp từ mọi pixel. CNN phù hợp với ảnh vì các pixel gần nhau mang thông tin không gian. Dịch chuyển không làm mạng bất biến tuyệt đối: padding, pooling và phần dense vẫn ảnh hưởng đáp ứng.

## 2. Convolution thủ công và kích thước

Các framework thường thực hiện **cross-correlation** (không lật kernel):

$$Y_{n,o,i,j}=b_o+\sum_{c,u,v}W_{o,c,u,v}X_{n,c,i+u-P,j+v-P}.$$

Với dilation $D=1$: $H_{out}=\lfloor(H+2P-K)/S\rfloor+1$. Số tham số convolution là $(K^2 C_{in}+1)C_{out}$. Trong bài, $K=3,P=1,S=1$ nên convolution giữ chiều rộng/cao. MaxPool 2×2 stride 2 giảm mỗi chiều một nửa.

Ví dụ 1D dưới đây tính đúng tổng tích; cần tự kiểm tra số học thay vì sao chép kết quả minh họa trong slide.

In [2]:
x = np.array([2., 1., 3.]); kernel = np.array([.5, -1., .5])
print('Tổng tích =', x @ kernel)
assert x @ kernel == 1.5

Tổng tích = 1.5


In [3]:
from src.numpy_cnn import Conv2D
rng = np.random.default_rng(42)
layer = Conv2D(1, 1, rng, kernel=3, padding=0)
layer.w[:] = np.array([[[[-1,0,1],[-1,0,1],[-1,0,1]]]], dtype='float32')
layer.b[:] = 0
image = np.arange(25, dtype='float32').reshape(1,1,5,5)
print('Ảnh đầu vào:'); print(image[0,0])
print('Đáp ứng bộ lọc cạnh:'); print(layer.forward(image, False)[0,0])

Ảnh đầu vào:
[[ 0.  1.  2.  3.  4.]
 [ 5.  6.  7.  8.  9.]
 [10. 11. 12. 13. 14.]
 [15. 16. 17. 18. 19.]
 [20. 21. 22. 23. 24.]]
Đáp ứng bộ lọc cạnh:
[[6. 6. 6.]
 [6. 6. 6.]
 [6. 6. 6.]]


## 3. Lan truyền ngược

Quy tắc dây chuyền đưa gradient từ loss về đầu vào. Với $G=\partial L/\partial Y$, gradient trọng số convolution cộng đóng góp ở mọi ảnh/vị trí. Mỗi pixel đầu vào nhận tổng gradient từ mọi cửa sổ chứa nó. Vì các cửa sổ chồng lấn, phép `col2im` phải **cộng dồn**, không ghi đè.

ReLU truyền gradient khi đầu vào dương; max pooling truyền về phần tử cực đại được chọn. Dense có $dW=X^T G$, $db=\sum G$, $dX=GW^T$. Cross-entropy kết hợp softmax có $\partial L/\partial z=(p-\text{onehot}(y))/N$.

Để ổn định số, trừ max của từng hàng logits trước khi tính exp. NumPy tự viết các đạo hàm và Adam; PyTorch dùng `backward`; TensorFlow dùng `GradientTape`.

In [4]:
from src.numpy_cnn import cross_entropy
logits = np.array([[1000.,1001.,999.]],dtype='float32')
loss, gradient = cross_entropy(logits,np.array([1]))
print('Loss hữu hạn:',loss,'; tổng gradient:',gradient.sum())

Loss hữu hạn: 0.40760600566864014 ; tổng gradient: -2.2351742e-08


## 4. Cải tiến CNN

**BatchNorm:** $\hat{x}=(x-\mu)/\sqrt{\sigma^2+\epsilon}$, đầu ra $\gamma\hat{x}+\beta$. Trong train dùng thống kê batch; trong eval dùng running statistics. Bài này chuẩn hóa ở ba vị trí convolution và cả sau Dense 64, trước ReLU.

**Residual:** $y=\mathrm{ReLU}(F(x)+x)$, trong đó $F$ là Conv+BN. Hai nhánh cùng kích thước. Gradient được cộng qua nhánh identity và nhánh học được. Đây là block residual nhỏ cho bài thực hành, không phải toàn bộ ResNet-18.

**Dropout:** trong train giữ mỗi phần tử với xác suất 0,75 rồi chia cho 0,75; trong eval không che phần tử. Mục đích là regularization, không có bảo đảm tăng accuracy cho mọi bộ dữ liệu.

Bản thử đầu tiên chưa có BatchNorm sau Dense học chậm trên CIFAR-100. Phiên bản cuối thêm BN tại đây dựa trên loss train/validation, rồi dùng thống nhất ở cả ba backend. Kết quả bản thử được giữ tại `experiments/pilot_v1/`.

In [5]:
from src.numpy_cnn import CNN
rows=[]
for name,cfg in DATASETS.items():
    for variant in ['baseline','improved']:
        model=CNN(cfg['channels'],cfg['size'],cfg['classes'],variant)
        rows.append(dict(dataset=name,variant=variant,parameters=model.parameter_count()))
display(pd.DataFrame(rows))
model=CNN(3,32,100,'improved')
x=np.zeros((2,3,32,32),dtype='float32')
shapes=[]
for i,layer in enumerate(model.layers):
    x=layer.forward(x,False)
    shapes.append(dict(index=i,layer=type(layer).__name__,output_shape=str(x.shape)))
display(pd.DataFrame(shapes))

,dataset,variant,parameters
0,mnist,baseline,52138
1,mnist,improved,54666
2,cifar10,baseline,67642
3,cifar10,improved,70170
4,cifar100,baseline,73492
5,cifar100,improved,76020


,index,layer,output_shape
0,0,Conv2D,"(2, 8, 32, 32)"
1,1,BatchNorm2D,"(2, 8, 32, 32)"
2,2,ReLU,"(2, 8, 32, 32)"
3,3,MaxPool2D,"(2, 8, 16, 16)"
4,4,Conv2D,"(2, 16, 16, 16)"
5,5,BatchNorm2D,"(2, 16, 16, 16)"
6,6,ReLU,"(2, 16, 16, 16)"
7,7,Residual,"(2, 16, 16, 16)"
8,8,MaxPool2D,"(2, 16, 8, 8)"
9,9,Flatten,"(2, 1024)"


## 5. Thiết kế thí nghiệm và câu hỏi cần giải thích

Baseline có 2 Conv + 2 Dense; improved có 3 Conv + 2 Dense, cộng 4 BatchNorm. Luôn ghi rõ quy ước đếm lớp thay vì chỉ gọi “CNN nhiều lớp”.

Cùng split, preprocessing, initialization và minibatch order giúp so sánh kiến trúc/triển khai. Chọn checkpoint bằng validation loss; test phục vụ đánh giá cuối. Hãy giải thích: vì sao Dense chiếm nhiều tham số, vì sao 100 lớp khó hơn 10 lớp, vì sao dropout làm train accuracy khó so trực tiếp với eval accuracy, và vì sao cải tiến có thể cần thêm epoch.

Tham khảo: tài liệu CNN của học phần; Chollet (2021), chương đánh giá mô hình, CNN và kiến trúc hiện đại; [BN](https://arxiv.org/abs/1502.03167), [ResNet](https://arxiv.org/abs/1512.03385), [Dropout](https://www.jmlr.org/papers/v15/srivastava14a.html).